<h1 style="color: #f686bd">Aprendizado Supervisionado Prático com Scikit-Learn</h1>

**Autor(a):** Carolina Nascimento

**Data:** 13/03/26

**Objetivo:** Neste notebook, vamos aplicar os conceitos de **Regressão** e **Classificação** que aprendemos na teoria usando a biblioteca de Machine Learning em Python: a `scikit-learn` (ou apenas `sklearn`). Vamos usar a nossa base de dados `07_online_learning_engagement_dataset.csv` dados sintéticos que representam o envolvimento e o comportamento de aprendizagem dos alunos em um ambiente de educação online. Inclui diversas características relacionadas a dados demográficos dos alunos, hábitos de estudo, atividade na plataforma e desempenho acadêmico. Nosso objetivo será criar dois modelos:
1. **Regressão:** Prever a nota que o aluno vai tirar na prova final (`nota_final`).
2. **Classificação:** Prever se o aluno irá **abandonar** ou não o curso.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

# Ferramentas do Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

## 1. Carregando e Preparando os Dados</h3>
 base de dados retirada de: [data](https://www.kaggle.com/datasets/ssssws/online-learning-engagement-dataset)

In [ ]:
# Carregando os dados
df = pd.read_csv('data/07_online_learning_engagement_dataset.csv')
display(df.head(10))

In [ ]:
# Dicionário de renomeação das colunas para português brasileiro
mapa_colunas = {
    'student_id': 'id_aluno',
    'age': 'idade',
    'gender': 'genero',
    'country': 'pais',
    'device_type': 'tipo_dispositivo',
    'internet_speed_mbps': 'velocidade_internet_mbps',
    'study_hours_weekly': 'horas_estudo_semanais',
    'login_frequency_weekly': 'frequencia_login_semanal',
    'avg_session_duration_min': 'duracao_media_sessao_min',
    'video_watch_time_min': 'tempo_video_min',
    'assignments_submitted': 'atividades_entregues',
    'forum_posts': 'posts_forum',
    'quiz_attempts': 'tentativas_quiz',
    'avg_quiz_score': 'media_quiz',
    'attendance_rate': 'taxa_presenca',
    'engagement_score': 'pontuacao_engajamento',
    'final_grade': 'nota_final',
    'dropout': 'abandono'
}

df = df.rename(columns=mapa_colunas)
display(df.head())

## 2. Análise Exploratória de Dados e Pré-Processamento

In [ ]:
df.describe() # Estatística Descritiva das colunas numéricas.

In [ ]:
df.info()

Como Modelos matemáticos não entendem texto, iremos transformar as colunas de strings em colunas em numericas, para isso vamos usar `LabelEncoder` do `sklearn`
Cada país, dispositivo ou genero, agora nas novas colunas vão ser interpretados como numeros.

> Essa não é a melhor solução de todas e existem problemas em arbitrariamente definir numeros para variaveis categóricas porem para fins de estudo vamos fazer dessa forma. Outra forma de fazer isso seria utilizando dummy (one-hot encoding). 

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['genero_labelencoder'] = le.fit_transform(df['genero'])
df['pais_labelencoder'] = le.fit_transform(df['pais'])
df['tipo_dispositivo_labelencoder'] = le.fit_transform(df['tipo_dispositivo'])

df

## 3. Mão na Massa 01: Regressão</h3>
Queremos prever o valor numérico exato da coluna `nota_final`. 

O primeiro passo é separar quem é nossa variável alvo (`y`) e quem são as características que usaremos para prever (`X`). Depois, vamos dividir os dados em **Treino (80%)** e **Teste (20%)**.

In [ ]:
# Variável Alvo (Target)
y = df['nota_final']

# Features (Variáveis preditoras) - Todas menos a nota final e as colunas strings que foram modificadas
X = df.drop(columns=['nota_final', 'genero', 'pais','tipo_dispositivo'])

# Separando em Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Temos {X_train.shape[0]} alunos para TREINAR o modelo.")
print(f"E guardamos {X_test.shape[0]} alunos inéditos para TESTAR o modelo.")

Agora vamos treinar dois modelos diferentes de Regressão: a clássica **Regressão Linear** e um poderoso **Random Forest Regressor**.

In [ ]:
# 1. Instanciar (Criar) os modelos
modelo_linear = LinearRegression()
modelo_arvore = RandomForestRegressor(random_state=42)

# 2. Treinar os modelos (o comando é sempre .fit())
modelo_linear.fit(X_train, y_train)
modelo_arvore.fit(X_train, y_train)

# 3. Fazer previsões com os dados de Teste (o comando é sempre .predict())
previsoes_linear = modelo_linear.predict(X_test)
previsoes_arvore = modelo_arvore.predict(X_test)

# Mostrando as 5 primeiras previsões vs as notas reais
df_resultado = pd.DataFrame({
    'Nota Real (Gabarito)': y_test.head(5),
    'Previsto Reg. Linear': previsoes_linear[:5].round(1),
    'Previsto Random Forest': previsoes_arvore[:5].round(1)
})
display(df_resultado)

Qual modelo foi melhor? Vamos avaliar usando o **MAE (Mean Absolute Error — Erro Absoluto Médio)**, que nos diz, em média, por quantos pontos nosso modelo errou a nota do aluno.

In [ ]:
erro_linear = mean_absolute_error(y_test, previsoes_linear)
erro_arvore = mean_absolute_error(y_test, previsoes_arvore)

print(f"A Regressão Linear errou a nota, em média, por: {erro_linear:.2f} pontos")
print(f"O Random Forest errou a nota, em média, por: {erro_arvore:.2f} pontos")

# O Random Forest costuma ser bem mais preciso que a Regressão Linear, 
# mas os dois são ótimos exemplos práticos.

Vamos visualizar em gráficos como os modelos se sairam.

In [ ]:
import seaborn as sns
import pandas as pd

erro_linear = abs(y_test - previsoes_linear)
erro_arvore = abs(y_test - previsoes_arvore)

df_erros = pd.DataFrame({
    "Regressão Linear": erro_linear,
    "Random Forest": erro_arvore
})

sns.boxplot(data=df_erros)

plt.ylabel("Erro absoluto")
plt.title("Distribuição do erro dos modelos")

plt.show()

In [ ]:
import pandas as pd

importances = modelo_arvore.feature_importances_

df_importancia = pd.DataFrame({
    "variavel": X_train.columns,
    "importancia": importances
}).sort_values("importancia", ascending=False)

df_importancia.head(10).plot.bar(x="variavel", y="importancia", legend=False)

plt.title("Variáveis mais importantes para prever a nota")
plt.ylabel("Importância")

plt.show()

In [ ]:
plt.figure(figsize=(15,10))
sns.regplot(data=df, x='media_quiz', y='nota_final', marker='o', ci=False,
            scatter_kws={"color":'navy', 'alpha':0.9, 's':220},
            line_kws={"color":'grey', 'linewidth': 5})
plt.title('Valores Reais e Fitted Values (Modelo de Regressão)', fontsize=30)
plt.xlabel('media_quiz', fontsize=24)
plt.ylabel('nota_final', fontsize=24)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.xlim(0, 150)
plt.ylim(0, 100)
plt.legend(['Valores Reais', 'Fitted Values'], fontsize=24, loc='upper left')
plt.show()


## 3. Mão na Massa 02: Classificação
Agora vamos mudar a abordagem. Não queremos mais saber a *nota exata*, queremos saber a pessoa vai ou não desistir?

In [ ]:

# Agora nossa variável alvo é a categoria 'y_class = df['abandono']

y_class = df['abandono']

# E as variáveis preditoras são as mesmas, 
# mas temos que retirar a 'Nota_na_Prova' para não dar a resposta de bandeja pro modelo!
X_class = df.drop(columns=['nota_final', 'genero', 'pais','tipo_dispositivo', 'abandono'])

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_class, y_class, test_size=0.2, random_state=42)

In [ ]:
# Treinando um Classificador Random Forest
modelo_classificador = RandomForestClassifier(random_state=42)
modelo_classificador.fit(X_train_c, y_train_c)

# Fazendo previsões
previsoes_class = modelo_classificador.predict(X_test_c)

# Avaliando com Acurácia (Porcentagem de acertos)
acuracia = accuracy_score(y_test_c, previsoes_class)
print(f"O modelo acertou {acuracia * 100:.1f}% das classificações (abandono/não abandono)!")

A **Matriz de Confusão** é a melhor forma de visualizar os erros e acertos de um modelo de classificação:

In [ ]:
matriz = confusion_matrix(y_test_c, previsoes_class)

plt.figure(figsize=(6,5))

sns.heatmap(
    matriz,
    annot=True,
    fmt='d',
    cmap='RdPu',
    xticklabels=['Não abandono','Abandono'],
    yticklabels=['Não abandono','Abandono']
)

plt.title("Matriz de Confusão")
plt.xlabel("Previsão do modelo")
plt.ylabel("Valor real")

plt.show()

In [ ]:
df['abandono'].value_counts().plot(
    kind='bar'
)

plt.title("Distribuição de abandono no dataset")
plt.xlabel("Abandono")
plt.ylabel("Quantidade de alunos")

plt.xticks([0,1], ['Não abandono','Abandono'])

plt.show()

In [ ]:
importances = modelo_classificador.feature_importances_

df_importancia = pd.DataFrame({
    "variavel": X_train_c.columns,
    "importancia": importances
}).sort_values("importancia", ascending=False)

plt.figure(figsize=(8,5))

sns.barplot(
    data=df_importancia.head(10),
    x="importancia",
    y="variavel"
)

plt.title("Variáveis mais importantes para prever abandono")

plt.show()